# Step 19.2 — Train / Validation Loop

本节覆盖 periodic validation loss 与 train/eval behavior，沿用19.1六项候选、完整MiniGPT、sampler、标量loss、Adam和固定MLX v0.32.2接口。

19.1提交 `3d2e95606daf1b5d789599c3d29fd0696240be26` 已独立核心静态验收PASS，实际修正0/2。这是材料验收，不是学员掌握或本机已有语料的证明。

所有代码单元均未执行且无输出。本节初始修正0/2；完成即停在19.2，不进入19.3。

## 1. 自包含依赖与候选配置

工作副本必须恢复所有导入、19.1候选、数据读取/切分/sampler、完整MiniGPT定义、标量loss、验证helper与循环。不能依赖旧kernel或假定本地语料已存在。缓存默认 `data/tinyshakespeare/input.txt`；下载入口必须单独显式启用，默认关闭。项目固定 `mlx==0.32.2`。

候选保持：width=64、layers=2、heads=4、block_size=64、batch_size=4、learning_rate=3e-4，`max_seq_len=block_size`。真实V必须由实际字符表获得。

In [ ]:
import math
import random
from pathlib import Path
from urllib.request import urlopen
import mlx.core as mx
import mlx.nn as nn
import mlx.optimizers as optim

baseline_candidate = {"model_width":64, "layers":2, "heads":4, "block_size":64, "batch_size":4, "learning_rate":3e-4}
loop_plan = {"num_updates":4, "eval_every":2, "val_batches":2}
seeds = {"model":2028, "train":2026, "validation":2027}

def make_model_kwargs(c, vocab_size):
    return {"vocab_size":vocab_size, "dims":c["model_width"], "num_heads":c["heads"], "num_blocks":c["layers"], "max_seq_len":c["block_size"]}
def make_sampler_kwargs(c): return {"batch_size":c["batch_size"], "block_size":c["block_size"]}
def make_optimizer_kwargs(c): return {"learning_rate":c["learning_rate"], "betas":[0.9,0.999], "eps":1e-8, "bias_correction":True} # explicit course choice; MLX default False

## 2. 数据边界与Sampler

先将连续token stream按90/10切分，再将 `train_tokens` 与 `val_tokens` 分别传给sampler。每个样本取T+1个连续token构造X/Y，各为(B,T)；窗口不跨split边界。Y已shift，loss中不再shift。两侧都必须至少有T+1=65个token。

训练使用持久Python RNG；验证每次使用独立固定seed重建RNG，以复用相同抽样窗口，不消耗训练RNG。它是抽样估计，不代表完整遍历validation split。

In [ ]:
SOURCE_URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
def acquire_tiny_shakespeare(path, source_url=SOURCE_URL):
    path=Path(path)
    if path.is_file(): return path
    path.parent.mkdir(parents=True, exist_ok=True)
    with urlopen(source_url, timeout=30) as response: raw=response.read()
    tmp=path.with_suffix('.tmp'); tmp.write_bytes(raw); tmp.replace(path); return path
def load_cached_character_corpus(path):
    path=Path(path)
    if not path.is_file(): raise FileNotFoundError(f"Cache missing: {path.resolve()}; correct path or explicitly acquire it.")
    text=path.read_text(encoding='utf-8'); chars=sorted(set(text)); stoi={ch:i for i,ch in enumerate(chars)}; itos={i:ch for i,ch in enumerate(chars)}
    return text, chars, stoi, itos, mx.array([stoi[ch] for ch in text], dtype=mx.int32)
def split_train_validation(stream, fraction=0.90):
    if len(stream.shape)!=1 or stream.shape[0]<2 or not 0<fraction<1: raise ValueError('Expected 1-D stream and 0<fraction<1')
    s=int(stream.shape[0]*fraction)
    if s<=0 or s>=stream.shape[0]: raise ValueError('Empty split')
    return stream[:s],stream[s:],s
def get_batch(data, batch_size, block_size, rng):
    if len(data.shape)!=1 or type(batch_size) is not int or batch_size<1 or type(block_size) is not int or block_size<1: raise ValueError('Invalid sampler inputs')
    n=data.shape[0]
    if n<block_size+1: raise ValueError('Split needs block_size+1 tokens')
    starts=[rng.randrange(0,n-block_size) for _ in range(batch_size)]; xs=[]; ys=[]
    for s in starts:
        chunk=data[s:s+block_size+1]; xs.append(chunk[:-1]); ys.append(chunk[1:])
    return mx.stack(xs,axis=0),mx.stack(ys,axis=0),starts

## 3. 完整模型与loss

模型定义保持既有15.4母版的CausalMultiHeadAttention、FeedForwardMLP、TransformerBlock与MiniGPT。恢复时应将这四个类的完整定义复制到本节工作副本，不使用SharedLogitModel代替；架构保持Pre-Norm RMSNorm、因果attention、C→4C→C MLP、独立Block、final RMSNorm和untied LM head。

Loss使用raw logits (B,T,V)，flatten为(B*T,V)与(B*T,)，逐位置cross-entropy后mean为标量()。训练调用nn.value_and_grad；验证直接调用loss_fn。

In [ ]:
import math
import mlx.core as mx
import mlx.nn as nn


class CausalMultiHeadAttention(nn.Module):
    def __init__(self, dims: int, num_heads: int):
        super().__init__()

        if dims <= 0:
            raise ValueError('dims must be positive')
        if num_heads <= 0:
            raise ValueError('num_heads must be positive')
        if dims % num_heads != 0:
            raise ValueError('dims must be divisible by num_heads')

        self.dims = dims
        self.num_heads = num_heads
        self.head_dim = dims // num_heads
        self.scale = 1.0 / math.sqrt(self.head_dim)  # Python float, not trainable

        self.qkv_proj = nn.Linear(
            input_dims=dims,
            output_dims=3 * dims,
            bias=False,
        )
        self.c_proj = nn.Linear(
            input_dims=dims,
            output_dims=dims,
            bias=False,
        )

    def __call__(self, x: mx.array) -> mx.array:
        if len(x.shape) != 3:
            raise ValueError('attention input must have shape (B,T,C)')

        B, T, C = x.shape
        if C != self.dims:
            raise ValueError('attention input width must equal dims')

        H = self.num_heads
        D = self.head_dim

        qkv = self.qkv_proj(x)  # (B,T,3C)
        q, k, v = mx.split(qkv, indices_or_sections=3, axis=-1)

        q = mx.reshape(q, (B, T, H, D))
        k = mx.reshape(k, (B, T, H, D))
        v = mx.reshape(v, (B, T, H, D))

        q = mx.swapaxes(q, 1, 2)  # (B,H,T,D)
        k = mx.swapaxes(k, 1, 2)
        v = mx.swapaxes(v, 1, 2)

        scores = mx.matmul(
            q,
            mx.swapaxes(k, -1, -2),
        ) * self.scale  # (B,H,T,T)

        # IMPORTANT: structural mask is local to forward.
        # It is NOT stored as self.causal_mask.
        future = mx.triu(mx.ones((T, T)), k=1) > 0
        mask = mx.where(
            future,
            mx.array(float('-inf'), dtype=scores.dtype),
            mx.array(0.0, dtype=scores.dtype),
        )

        weights = mx.softmax(scores + mask, axis=-1)
        head_out = mx.matmul(weights, v)  # (B,H,T,D)

        token_major = mx.swapaxes(head_out, 1, 2)  # (B,T,H,D)
        merged = mx.reshape(token_major, (B, T, C))
        return self.c_proj(merged)  # (B,T,C)


class FeedForwardMLP(nn.Module):
    def __init__(self, dims: int):
        super().__init__()
        self.fc1 = nn.Linear(dims, 4 * dims, bias=True)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(4 * dims, dims, bias=True)

    def __call__(self, x: mx.array) -> mx.array:
        return self.fc2(self.act(self.fc1(x)))


class TransformerBlock(nn.Module):
    def __init__(self, dims: int, num_heads: int):
        super().__init__()
        self.norm1 = nn.RMSNorm(dims=dims)
        self.attn = CausalMultiHeadAttention(dims=dims, num_heads=num_heads)
        self.norm2 = nn.RMSNorm(dims=dims)
        self.mlp = FeedForwardMLP(dims=dims)

    def __call__(self, x: mx.array) -> mx.array:
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x


# Definitions only. This master notebook does not execute them.


import math
import mlx.core as mx
import mlx.nn as nn


class CausalMultiHeadAttention(nn.Module):
    def __init__(self, dims: int, num_heads: int):
        super().__init__()

        if dims <= 0:
            raise ValueError('dims must be positive')
        if num_heads <= 0:
            raise ValueError('num_heads must be positive')
        if dims % num_heads != 0:
            raise ValueError('dims must be divisible by num_heads')

        self.dims = dims
        self.num_heads = num_heads
        self.head_dim = dims // num_heads
        self.scale = 1.0 / math.sqrt(self.head_dim)  # Python float, not trainable

        self.qkv_proj = nn.Linear(
            input_dims=dims,
            output_dims=3 * dims,
            bias=False,
        )
        self.c_proj = nn.Linear(
            input_dims=dims,
            output_dims=dims,
            bias=False,
        )

    def __call__(self, x: mx.array) -> mx.array:
        if len(x.shape) != 3:
            raise ValueError('attention input must have shape (B,T,C)')

        B, T, C = x.shape
        if C != self.dims:
            raise ValueError('attention input width must equal dims')

        H = self.num_heads
        D = self.head_dim

        qkv = self.qkv_proj(x)  # (B,T,3C)
        q, k, v = mx.split(qkv, indices_or_sections=3, axis=-1)

        q = mx.reshape(q, (B, T, H, D))
        k = mx.reshape(k, (B, T, H, D))
        v = mx.reshape(v, (B, T, H, D))

        q = mx.swapaxes(q, 1, 2)  # (B,H,T,D)
        k = mx.swapaxes(k, 1, 2)
        v = mx.swapaxes(v, 1, 2)

        scores = mx.matmul(
            q,
            mx.swapaxes(k, -1, -2),
        ) * self.scale  # (B,H,T,T)

        # IMPORTANT: structural mask is local to forward.
        # It is NOT stored as self.causal_mask.
        future = mx.triu(mx.ones((T, T)), k=1) > 0
        mask = mx.where(
            future,
            mx.array(float('-inf'), dtype=scores.dtype),
            mx.array(0.0, dtype=scores.dtype),
        )

        weights = mx.softmax(scores + mask, axis=-1)
        head_out = mx.matmul(weights, v)  # (B,H,T,D)

        token_major = mx.swapaxes(head_out, 1, 2)  # (B,T,H,D)
        merged = mx.reshape(token_major, (B, T, C))
        return self.c_proj(merged)  # (B,T,C)


class FeedForwardMLP(nn.Module):
    def __init__(self, dims: int):
        super().__init__()
        self.fc1 = nn.Linear(dims, 4 * dims, bias=True)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(4 * dims, dims, bias=True)

    def __call__(self, x: mx.array) -> mx.array:
        return self.fc2(self.act(self.fc1(x)))


class TransformerBlock(nn.Module):
    def __init__(self, dims: int, num_heads: int):
        super().__init__()
        self.norm1 = nn.RMSNorm(dims=dims)
        self.attn = CausalMultiHeadAttention(dims=dims, num_heads=num_heads)
        self.norm2 = nn.RMSNorm(dims=dims)
        self.mlp = FeedForwardMLP(dims=dims)

    def __call__(self, x: mx.array) -> mx.array:
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x


# Definitions only. This master notebook does not execute them.


import math
import mlx.core as mx
import mlx.nn as nn


class CausalMultiHeadAttention(nn.Module):
    def __init__(self, dims: int, num_heads: int):
        super().__init__()

        if dims <= 0:
            raise ValueError('dims must be positive')
        if num_heads <= 0:
            raise ValueError('num_heads must be positive')
        if dims % num_heads != 0:
            raise ValueError('dims must be divisible by num_heads')

        self.dims = dims
        self.num_heads = num_heads
        self.head_dim = dims // num_heads
        self.scale = 1.0 / math.sqrt(self.head_dim)  # Python float, not trainable

        self.qkv_proj = nn.Linear(
            input_dims=dims,
            output_dims=3 * dims,
            bias=False,
        )
        self.c_proj = nn.Linear(
            input_dims=dims,
            output_dims=dims,
            bias=False,
        )

    def __call__(self, x: mx.array) -> mx.array:
        if len(x.shape) != 3:
            raise ValueError('attention input must have shape (B,T,C)')

        B, T, C = x.shape
        if C != self.dims:
            raise ValueError('attention input width must equal dims')

        H = self.num_heads
        D = self.head_dim

        qkv = self.qkv_proj(x)  # (B,T,3C)
        q, k, v = mx.split(qkv, indices_or_sections=3, axis=-1)

        q = mx.reshape(q, (B, T, H, D))
        k = mx.reshape(k, (B, T, H, D))
        v = mx.reshape(v, (B, T, H, D))

        q = mx.swapaxes(q, 1, 2)  # (B,H,T,D)
        k = mx.swapaxes(k, 1, 2)
        v = mx.swapaxes(v, 1, 2)

        scores = mx.matmul(
            q,
            mx.swapaxes(k, -1, -2),
        ) * self.scale  # (B,H,T,T)

        # IMPORTANT: structural mask is local to forward.
        # It is NOT stored as self.causal_mask.
        future = mx.triu(mx.ones((T, T)), k=1) > 0
        mask = mx.where(
            future,
            mx.array(float('-inf'), dtype=scores.dtype),
            mx.array(0.0, dtype=scores.dtype),
        )

        weights = mx.softmax(scores + mask, axis=-1)
        head_out = mx.matmul(weights, v)  # (B,H,T,D)

        token_major = mx.swapaxes(head_out, 1, 2)  # (B,T,H,D)
        merged = mx.reshape(token_major, (B, T, C))
        return self.c_proj(merged)  # (B,T,C)


class FeedForwardMLP(nn.Module):
    def __init__(self, dims: int):
        super().__init__()
        self.fc1 = nn.Linear(dims, 4 * dims, bias=True)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(4 * dims, dims, bias=True)

    def __call__(self, x: mx.array) -> mx.array:
        return self.fc2(self.act(self.fc1(x)))


class TransformerBlock(nn.Module):
    def __init__(self, dims: int, num_heads: int):
        super().__init__()
        self.norm1 = nn.RMSNorm(dims=dims)
        self.attn = CausalMultiHeadAttention(dims=dims, num_heads=num_heads)
        self.norm2 = nn.RMSNorm(dims=dims)
        self.mlp = FeedForwardMLP(dims=dims)

    def __call__(self, x: mx.array) -> mx.array:
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x


# Definitions only. This master notebook does not execute them.


class MiniGPT(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        dims: int,
        num_heads: int,
        num_blocks: int,
        max_seq_len: int,
    ):
        super().__init__()

        if vocab_size <= 0:
            raise ValueError('vocab_size must be positive')
        if dims <= 0:
            raise ValueError('dims must be positive')
        if num_heads <= 0:
            raise ValueError('num_heads must be positive')
        if num_blocks <= 0:
            raise ValueError('num_blocks must be positive')
        if max_seq_len <= 0:
            raise ValueError('max_seq_len must be positive')
        if dims % num_heads != 0:
            raise ValueError('dims must be divisible by num_heads')

        # Python metadata: these are not trainable mx.array parameters.
        self.vocab_size = vocab_size
        self.dims = dims
        self.max_seq_len = max_seq_len

        # Learned lookup tables.
        self.token_embedding = nn.Embedding(
            num_embeddings=vocab_size,
            dims=dims,
        )
        self.position_embedding = nn.Embedding(
            num_embeddings=max_seq_len,
            dims=dims,
        )

        # A list of distinct Module instances; MLX recursively tracks them.
        self.blocks = [
            TransformerBlock(
                dims=dims,
                num_heads=num_heads,
            )
            for _ in range(num_blocks)
        ]

        self.final_norm = nn.RMSNorm(dims=dims)
        self.lm_head = nn.Linear(
            input_dims=dims,
            output_dims=vocab_size,
            bias=False,
        )

    def __call__(self, token_ids: mx.array) -> mx.array:
        if len(token_ids.shape) != 2:
            raise ValueError('token_ids must have shape (B,T)')

        B, T = token_ids.shape

        if T < 1:
            raise ValueError('sequence length T must be at least 1')
        if T > self.max_seq_len:
            raise ValueError(
                f'sequence length {T} exceeds max_seq_len {self.max_seq_len}'
            )

        # Forward-local structural indices: not Module parameters.
        position_ids = mx.arange(T)  # (T,)

        token_features = self.token_embedding(token_ids)        # (B,T,C)
        position_features = self.position_embedding(position_ids)  # (T,C)
        x = token_features + position_features                  # (B,T,C)

        for block in self.blocks:
            x = block(x)  # invariant: (B,T,C)

        x = self.final_norm(x)  # (B,T,C)
        logits = self.lm_head(x)  # (B,T,V)

        expected_shape = (B, T, self.vocab_size)
        if logits.shape != expected_shape:
            raise RuntimeError(
                f'logits shape {logits.shape} != expected {expected_shape}'
            )

        return logits


# Definition only. No MiniGPT forward is executed in this master notebook.


## 4. Train/eval、梯度与求值

`model.train()`设置training mode；`model.eval()`等价于`model.train(False)`。它不关闭梯度、不冻结参数、不自动完成前向。梯度由是否调用梯度变换决定，更新由是否调用optimizer决定。

当前MiniGPT没有Dropout、BatchNorm或读取`self.training`的前向分支；现有Embedding/Linear/GELU/RMSNorm不按模式改变公式。因此相同参数与输入下，单纯切换模式不改变本模型的前向数学定义。

`mx.eval`要求选定数组及依赖完成并返回None；它不是模式切换，也不等于关闭梯度。验证helper不接收optimizer、不调用nn.value_and_grad，因此没有参数更新路径。

In [ ]:
def estimate_validation_loss(model, val_tokens, batch_size, block_size, num_batches, seed):
    if type(num_batches) is not int or num_batches<1: raise ValueError('num_batches must be positive int')
    rng=random.Random(seed); was_training=model.training; model.eval()
    try:
        total_loss=0.0; total_positions=0
        for _ in range(num_batches):
            Xv,Yv,_=get_batch(val_tokens,batch_size,block_size,rng)
            loss=loss_fn(model,Xv,Yv) # no gradient transform, no optimizer
            mx.eval(loss); n=Yv.shape[0]*Yv.shape[1]
            total_loss += loss.item()*n; total_positions += n
        return total_loss/total_positions
    finally:
        model.train(was_training)

## 5. 周期性验证与参数版本

第k次训练调用返回的是更新前的 `L_train(θ[k−1]; B[k])`；更新后模型持有θ[k]，Adam状态也前进一步。到验证周期时，所有validation batch都使用同一当前参数θ[k]，得到`L_val(θ[k])`。不能把训练步返回的旧参数loss当成更新后loss。

若第j批mean loss为Lⱼ、监督位置数为nⱼ，合并平均为 `sum(nⱼ Lⱼ)/sum(nⱼ)`。本节固定B/T，因此每批nⱼ=256，等权batch mean的平均与位置加权平均等价。纸面例：两个假设mean为1.2和1.8，平均1.5；这不是模型结果。

In [ ]:
# One matched model/optimizer/compiled-step setup. This cell is not executed during preparation.
model_kwargs = make_model_kwargs(baseline_candidate, vocab_size=vocab_size)
sampler_kwargs = make_sampler_kwargs(baseline_candidate)
adam_kwargs = make_optimizer_kwargs(baseline_candidate)
mx.random.seed(seeds["model"])
model = MiniGPT(**model_kwargs)
model.train()
loss_and_grad_fn = nn.value_and_grad(model, loss_fn)
optimizer = optim.Adam(**adam_kwargs)
optimizer.init(model.trainable_parameters())
training_state = [model.state, optimizer.state]
train_rng = random.Random(seeds["train"])
mx.eval(training_state)
def train_graph(X, Y):
    loss_before_update, grads = loss_and_grad_fn(model, X, Y)
    optimizer.update(model, grads)
    return loss_before_update
compiled_train_step = mx.compile(train_graph, inputs=training_state, outputs=training_state, shapeless=False)


In [ ]:
def run_training_with_validation(model, optimizer, train_step, training_state,
                                 train_tokens, val_tokens, train_rng, batch_size, block_size,
                                 num_updates, eval_every, val_batches, val_seed):
    if type(num_updates) is not int or num_updates<1 or type(eval_every) is not int or eval_every<1: raise ValueError('Invalid loop plan')
    model.train(); mx.eval(training_state); completed=int(optimizer.state['step'].item())
    initial_val=estimate_validation_loss(model,val_tokens,batch_size,block_size,val_batches,val_seed)
    history=[{'completed_updates':completed,'train_loss_before_update':None,'validation_loss_current':initial_val}]
    for local_update in range(1,num_updates+1):
        model.train(); X,Y,_=get_batch(train_tokens,batch_size,block_size,train_rng)
        train_loss_before_update=train_step(X,Y)
        mx.eval(train_loss_before_update,training_state); completed += 1
        train_value=train_loss_before_update.item(); val_value=None
        if completed%eval_every==0 or local_update==num_updates:
            val_value=estimate_validation_loss(model,val_tokens,batch_size,block_size,val_batches,val_seed)
        history.append({'completed_updates':completed,'train_loss_before_update':train_value,'validation_loss_current':val_value})
    return history

## 6. 编译状态与未来使用顺序

将模型、nn.value_and_grad、Adam、`[model.state, optimizer.state]`原容器引用和compiled callable作为同一匹配setup。训练图只执行forward/loss、梯度与Adam update并返回更新前loss；采样、模式切换、日志和mx.eval留在图外。编译不等于求值，也不保证提速或省内存。

未来work copy操作顺序：恢复本Notebook中的完整MiniGPT类定义；定位真实缓存，必要时主动启用可选获取；执行本地bootstrap；构造model与optimizer并初始化状态；定义训练图和compiled wrapper；最后才执行训练循环。当前没有loss记录、训练结果或验证结果。

累计修正：16.1/16.2各1/2；16.3—16.5、17.1—17.5、18.1—18.5及19.1各0/2；19.2初始0/2。只编写与发布材料，本轮不执行Notebook、训练、验证、测试、采样、测量或语料读写。停在19.2，不进入19.3；后续逐小节至大Step20完成后停止，不进入21。